# Dev Notebook for hf-2.0 Branch

In [1]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from meter.modules.heads import Pooler

from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule

from torch.optim import AdamW

from transformers import AutoConfig
from transformers import ElectraTokenizer, ViltFeatureExtractor
from transformers import AutoProcessor, AutoImageProcessor, AutoTokenizer
from transformers import AutoModel, AutoModelForSequenceClassification

# from refcoco_utils import get_bounded_subimage
# from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

In [2]:
def _loss_names(d):
    ret = {
        "itm": 0,
        "mlm": 0,
        "mpp": 0,
        "vqa": 0,
        "vcr": 0,
        "vcr_qar": 0,
        "nlvr2": 0,
        "irtr": 0,
        "contras": 0,
        "snli": 0,
        "ref": 0,
        "mrpc" : 1,
        "rte" : 0,
        'wnli' : 0,
        'sst2' : 0,
        'qqp' : 0,
        'qnli' : 0,
        'mnli' : 0,
        'cola' : 0,
        'cifar10' : 0
    }
    ret.update(d)
    return ret

config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 42,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["snli"],
    # 'loss_names' : _loss_names({"itm": 1, "mlm": 1}),
    'loss_names' : _loss_names({"snli": 1}),
    "batch_size" : 1,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.
    "model_type" : 'two-tower',
    
    # One-Tower Settings
    "random_init_encoder" : False,
    "encoder" : "facebook/deit-tiny-patch16-224",
    'encoder_type' : 'image',
    # Transformer Setting
    # 'vit' : "vit_base_patch32_384",
    'hidden_size' : 192,
    'num_heads' : 12,
    'num_layers' : 12,
    'mlp_ratio' : 4,
    'drop_rate' : 0.1,
    

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    'pooler_type' : 'double', 

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Architecture Setting
    "two_tower" : False,
    "multi_modal_encoder" : 'dandelin/vilt-b32-mlm',
    
    
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 5e-5,
    "weight_decay" : 0.0,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 0,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    
    'text_only' : False,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    # "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 2,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    'load_path' : '',
    # "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    # "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}


## DataModule

In [3]:
dm = MTDataModule(config, dist=False)

In [4]:
model = METERTransformerSS(config)

Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
model

METERTransformerSS(
  (cross_modal_text_transform): Linear(in_features=256, out_features=256, bias=True)
  (cross_modal_image_transform): Linear(in_features=192, out_features=256, bias=True)
  (cross_modal_image_layers): ModuleList(
    (0-5): 6 x BertCrossLayer(
      (attention): BertAttention(
        (self): BertSelfAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (output): BertSelfOutput(
          (dense): Linear(in_features=256, out_features=256, bias=True)
          (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (crossattention): BertAttention(
        (self): BertSelfAttention(
          (query): Linear(in_features=256, out_features=256, bias=

In [6]:
dm.prepare_data()
dm.setup('train')

In [7]:
dl = dm.train_dataloader()

In [8]:
batch = next(iter(dl))

You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenize

In [9]:
batch

{'labels': [0, 2],
 'table_name': ['snli_train', 'snli_train'],
 'text': ['Two old men robbing a convenience store.',
  'Two humans in a store.'],
 'image': [tensor([[[[ 1.1529,  0.7248,  0.6906,  ...,  0.5193,  0.2111,  0.7248],
            [ 0.4166, -0.0116,  1.1872,  ...,  0.4508,  0.2624,  0.6392],
            [-0.5424, -0.7479, -0.3369,  ...,  0.5536,  0.2624,  0.7591],
            ...,
            [ 0.5022,  0.5364,  0.5364,  ..., -1.1932, -1.1418, -1.1589],
            [ 0.7762,  0.7591,  0.7419,  ..., -0.8849, -0.8678, -0.9192],
            [ 0.9474,  0.8789,  0.8618,  ..., -0.0801,  0.0569,  0.0056]],
  
           [[ 0.9930,  0.6604,  0.7654,  ...,  0.8354,  0.4678,  0.9230],
            [ 0.5028,  0.1527,  1.2731,  ...,  0.7304,  0.5203,  0.8880],
            [-0.5651, -0.5301, -0.1625,  ...,  0.8354,  0.5028,  1.0280],
            ...,
            [ 0.5903,  0.6254,  0.6254,  ..., -1.1429, -1.1078, -1.0903],
            [ 0.8704,  0.8529,  0.8354,  ..., -0.7052, -0.7052, -0

In [10]:
text_encoder = model.text_transformer
image_encoder = model.image_encoder
cross_modal_image_transform = model.cross_modal_image_transform
cross_modal_text_transform = model.cross_modal_text_transform
cross_modal_text_transform

Linear(in_features=256, out_features=256, bias=True)

In [11]:
text_hidden_state = text_encoder( input_ids = batch['text_ids'], attention_mask = batch['text_masks'])[0]
text_hidden_state = cross_modal_text_transform(text_hidden_state)
text_hidden_state.shape

torch.Size([2, 128, 256])

In [12]:
image_hidden_state = image_encoder(batch['image'][0])[0]
image_hidden_state = cross_modal_image_transform(image_hidden_state)
image_hidden_state.shape

torch.Size([2, 197, 256])

In [13]:
# model.text_transformer.get_extended_attention_mask(image_masks, image_masks.size())

In [15]:
# model.fusion_encoder.cross_modal_image_layers[0]

## Use Huggingface for CrossLayers

In [16]:
from meter.modules.bert_model import BertCrossLayer
from transformers.models.bert.modeling_bert import BertLayer
from transformers.models.bert.configuration_bert import BertConfig

import torch.nn as nn

In [17]:
bert_config = BertConfig(
            vocab_size=config["vocab_size"],
            hidden_size=config["cross_layer_hidden_size"],
            num_attention_heads=config["num_cross_layer_heads"],
            intermediate_size=config["cross_layer_hidden_size"] * config["cross_layer_mlp_ratio"],
            max_position_embeddings=config["max_text_len"],
            hidden_dropout_prob=config["cross_layer_drop_rate"],
            attention_probs_dropout_prob=config["cross_layer_drop_rate"],
)
bert_config

BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 256,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 128,
  "model_type": "bert",
  "num_attention_heads": 4,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.36.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [18]:
bert_config = BertConfig(config, add_cross_attention=True, is_decoder=True, 
                         hidden_size=config['cross_layer_hidden_size'],
                        num_attention_heads=4)
bert_config

BertConfig {
  "add_cross_attention": true,
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 256,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": true,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 4,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.36.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": {
    "batch_size": 1,
    "cross_layer_drop_rate": 0.1,
    "cross_layer_hidden_size": 256,
    "cross_layer_mlp_ratio": 4,
    "data_root": "/home/claytonfields/nlp/code/meter/data/arrow",
    "datasets": [
      "snli"
    ],
    "decay_power": 1,
    "draw_false_image": 1,
    "draw_false_text": 0,
    "drop_rate": 0.1,
    "encoder": "facebook/deit-tiny-patch16-224",
    "encoder_type": "image",
    "end_lr": 0,
    "exp_na

### BertLayer from models/bert_modeling.py

In [19]:
cross_modal_text_layers_bert = nn.ModuleList([BertLayer(bert_config) for _ in range(config['num_cross_layers'])])
cross_modal_image_layers_bert = nn.ModuleList([BertLayer(bert_config) for _ in range(config['num_cross_layers'])])

In [20]:
cross_modal_image_layers_bert

ModuleList(
  (0-5): 6 x BertLayer(
    (attention): BertAttention(
      (self): BertSelfAttention(
        (query): Linear(in_features=256, out_features=256, bias=True)
        (key): Linear(in_features=256, out_features=256, bias=True)
        (value): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (output): BertSelfOutput(
        (dense): Linear(in_features=256, out_features=256, bias=True)
        (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (crossattention): BertAttention(
      (self): BertSelfAttention(
        (query): Linear(in_features=256, out_features=256, bias=True)
        (key): Linear(in_features=256, out_features=256, bias=True)
        (value): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (output): BertSelfOutput(
        (dense): Linear(in_

In [21]:
do_mlm = ''
text_ids = batch[f"text_ids{do_mlm}"]
text_labels = batch[f"text_labels{do_mlm}"]
text_masks = batch["text_masks"]

In [22]:
text_masks.shape

torch.Size([2, 128])

In [23]:
input_shape = text_masks.size()
extend_text_masks = text_encoder.get_extended_attention_mask(text_masks, input_shape)
extend_text_masks.shape

torch.Size([2, 1, 1, 128])

In [24]:
image_embeds = image_encoder(batch['image'][0])[0]
image_embeds.shape

torch.Size([2, 197, 192])

In [25]:
image_hidden_state.shape

torch.Size([2, 197, 256])

In [26]:
image_masks = torch.ones((image_hidden_state.size(0), image_hidden_state.size(1)), dtype=torch.long)
extend_image_masks = text_encoder.get_extended_attention_mask(image_masks, image_masks.size())
extend_image_masks.shape

torch.Size([2, 1, 1, 197])

In [27]:
x, y = text_hidden_state, image_hidden_state
for text_layer, image_layer in zip(cross_modal_text_layers_bert, cross_modal_image_layers_bert):
    # x1 = text_layer(hidden_states=x, encoder_hidden_states=y, attention_mask=extend_text_masks, encoder_attention_mask=extend_image_masks)
    # y1 = image_layer(hidden_states=y, encoder_hidden_states=x, attention_mask=extend_image_masks, encoder_attention_mask=extend_text_masks)
    x1 = text_layer(hidden_states=x, encoder_hidden_states=y)
    y1 = image_layer(hidden_states=y, encoder_hidden_states=x)
    x, y = x1[0], y1[0]

text_feats, image_feats = x, y
# cls_feats_text = self.cross_modal_text_pooler(x)
# cls_feats_image = self.cross_modal_image_pooler(y)
# cls_feats = torch.cat([cls_feats_text, cls_feats_image], dim=-1)
print(text_feats)
image_feats

tensor([[[-0.6208, -2.0637,  0.9762,  ..., -0.6651, -0.7505, -0.3534],
         [-0.0554, -4.9096,  2.0689,  ...,  0.4383, -0.1782, -0.7158],
         [-0.7459, -3.5637,  1.6430,  ...,  0.2565, -0.5191, -1.2743],
         ...,
         [ 1.2563, -1.4043,  1.5388,  ...,  0.8242,  0.1268, -1.0592],
         [ 0.9133, -1.0632,  1.3884,  ...,  0.4051,  0.2226, -1.1205],
         [ 1.1250, -1.5334,  1.6495,  ...,  0.2010,  0.2321, -1.1146]],

        [[-0.5777, -2.6535,  1.1973,  ..., -0.9442, -1.0477,  0.0196],
         [-0.3547, -4.3242,  2.5835,  ...,  0.5026, -0.6279, -1.3252],
         [-0.4217, -3.7499,  2.0981,  ..., -0.1582,  0.1213, -0.9972],
         ...,
         [ 0.8877, -0.9316,  1.8994,  ...,  0.3933,  0.1196, -1.4667],
         [ 1.2802, -0.9204,  1.7367,  ...,  0.6264,  0.7245, -1.1665],
         [ 1.1842, -1.1267,  1.5926,  ...,  0.1654,  0.3873, -1.2011]]],
       grad_fn=<NativeLayerNormBackward0>)


tensor([[[ 0.7620, -0.4741,  0.0847,  ...,  0.5530,  0.4747, -2.7143],
         [ 0.0753,  1.0454, -0.7256,  ...,  1.6581,  0.3376, -1.5692],
         [ 0.0128, -0.7917, -0.4097,  ...,  1.3515,  0.4301, -0.6231],
         ...,
         [-0.4182, -0.1968, -0.9535,  ...,  0.2410, -0.1219, -1.8793],
         [-0.8913, -1.3057, -0.3937,  ...,  0.9548,  0.3710,  0.3224],
         [-0.3826, -0.7554,  1.3281,  ...,  0.7286,  1.2737, -0.4424]],

        [[ 0.4267, -0.5160,  0.0559,  ...,  0.8232,  0.5973, -1.7288],
         [ 0.0263,  0.7140, -0.9834,  ...,  2.1428,  0.3487, -1.0571],
         [ 0.2040, -0.5123, -0.9534,  ...,  1.6282,  0.6008, -0.8300],
         ...,
         [-0.4985, -0.6981, -0.4194,  ...,  0.3434,  0.0879, -1.8016],
         [-0.8540, -1.4315,  0.1302,  ...,  0.2185,  0.2500, -0.1154],
         [-0.8971, -0.8302,  0.9549,  ...,  0.7235,  1.2522, -0.8291]]],
       grad_fn=<NativeLayerNormBackward0>)

Why the extended masks?

What are they?

**Are they necessary?**

They appear to be necessary.

### LxmertCrossModalEncoder

In [28]:
from transformers.models.lxmert.modeling_lxmert import LxmertXLayer

In [29]:
cross_modal_layers_lx = nn.ModuleList([LxmertXLayer(bert_config) for _ in range(config['num_cross_layers'])])
# cross_modal_image_layers_lx = nn.ModuleList([LxmertXLayer(bert_config) for _ in range(config['num_cross_layers'])])

In [30]:
# x, y = text_hidden_state, image_hidden_state
# for text_layer, image_layer in zip(cross_modal_text_layers_lx, cross_modal_image_layers_lx):
#     # x1 = text_layer(hidden_states=x, encoder_hidden_states=y, attention_mask=extend_text_masks, encoder_attention_mask=extend_image_masks)
#     # y1 = image_layer(hidden_states=y, encoder_hidden_states=x, attention_mask=extend_image_masks, encoder_attention_mask=extend_text_masks)
#     x1 = text_layer(x, extend_text_masks, y, extend_image_masks)
#     y1 = image_layer(y, extend_image_masks,  x, extend_text_masks)
#     x, y = x1[0], y1[0]

# text_feats, image_feats = x, y
# # cls_feats_text = self.cross_modal_text_pooler(x)
# # cls_feats_image = self.cross_modal_image_pooler(y)
# # cls_feats = torch.cat([cls_feats_text, cls_feats_image], dim=-1)
# print(text_feats)
# image_feats

In [31]:
print(x.shape)
print(y.shape)

torch.Size([2, 128, 256])
torch.Size([2, 197, 256])


In [32]:
vision_hidden_states = ()
language_hidden_states = ()

In [33]:
for layer_module in cross_modal_layers_lx:
    
    x_outputs = layer_module(
        x, 
        extend_text_masks, 
        y, 
        extend_image_masks
    )
    lang_feats, visual_feats = x_outputs[:2]
#     vision_hidden_states = vision_hidden_states + (visual_feats,)
#     language_hidden_states = language_hidden_states + (lang_feats,)
    
# visual_encoder_outputs = (
#     vision_hidden_states,
#     vision_attentions if output_attentions else None,
# )
# lang_encoder_outputs = (
#     language_hidden_states,
#     language_attentions if output_attentions else None,
# )
print(lang_feats.shape)
visual_feats.shape

torch.Size([2, 128, 256])


torch.Size([2, 197, 256])

In [34]:
len(vision_hidden_states + (visual_feats,))

1

### TODO: Test both cross modal encoders!

In [35]:
from meter.modules.fusion_encoder import BertCrossModalEncoder, LxmertCrossModalEncoder

In [36]:
lx_encoder = LxmertCrossModalEncoder(config)

In [37]:
lx_encoder

LxmertCrossModalEncoder(
  (cross_modal_layers): ModuleList(
    (0-5): 6 x LxmertXLayer(
      (visual_attention): LxmertCrossAttentionLayer(
        (att): LxmertAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (output): LxmertAttentionOutput(
          (dense): Linear(in_features=256, out_features=256, bias=True)
          (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (lang_self_att): LxmertSelfAttentionLayer(
        (self): LxmertAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=T

In [38]:
output = lx_encoder(text_hidden_state,extend_text_masks,image_hidden_state,extend_image_masks)

In [39]:
output[0]

tensor([[-0.6261, -0.8405,  0.6083,  ...,  0.8300, -0.1651,  0.7554],
        [-0.2626, -0.7934,  0.5549,  ...,  0.8961, -0.2358,  0.7030]],
       grad_fn=<CatBackward0>)

In [40]:
image_hidden_state.shape

torch.Size([2, 197, 256])

In [41]:
from meter.modules.fusion_encoder import BertCrossModalEncoder

In [42]:
br_encoder = BertCrossModalEncoder(config)

In [43]:
br_output = br_encoder(text_hidden_state,image_hidden_state,extend_text_masks,extend_image_masks)

In [44]:
br_output[0]

tensor([[ 0.4442,  0.1093,  0.3470,  ..., -0.0118,  0.7573, -0.3732],
        [ 0.5872,  0.0199,  0.4143,  ...,  0.1117,  0.6405, -0.5549]],
       grad_fn=<CatBackward0>)

In [45]:
text_hidden_state

tensor([[[-0.2477,  0.0563, -0.0144,  ..., -0.0110, -0.0327,  0.0827],
         [-0.0201, -0.3472,  0.0321,  ...,  0.0995,  0.1278, -0.0653],
         [-0.0779,  0.0693,  0.1618,  ...,  0.1598, -0.0593, -0.0808],
         ...,
         [ 0.6188,  0.3879,  0.1211,  ...,  0.3049,  0.3392, -0.5552],
         [ 0.6355,  0.3825,  0.1168,  ...,  0.3080,  0.3246, -0.5201],
         [ 0.6400,  0.3673,  0.0967,  ...,  0.3032,  0.3148, -0.4942]],

        [[-0.1570,  0.0934, -0.0418,  ...,  0.0366, -0.1524,  0.0063],
         [ 0.0642, -0.3553,  0.0893,  ...,  0.1205,  0.1340, -0.1009],
         [ 0.0370,  0.0311,  0.1866,  ...,  0.0741,  0.1778, -0.2748],
         ...,
         [ 0.6413,  0.3969,  0.1238,  ...,  0.2639,  0.3195, -0.5872],
         [ 0.6528,  0.3912,  0.1178,  ...,  0.2663,  0.2972, -0.5573],
         [ 0.6455,  0.3922,  0.1036,  ...,  0.2582,  0.2861, -0.5472]]],
       grad_fn=<ViewBackward0>)

In [63]:
batch

{'labels': [0, 2],
 'table_name': ['snli_train', 'snli_train'],
 'text': ['Two old men robbing a convenience store.',
  'Two humans in a store.'],
 'image': [tensor([[[[ 1.1529,  0.7248,  0.6906,  ...,  0.5193,  0.2111,  0.7248],
            [ 0.4166, -0.0116,  1.1872,  ...,  0.4508,  0.2624,  0.6392],
            [-0.5424, -0.7479, -0.3369,  ...,  0.5536,  0.2624,  0.7591],
            ...,
            [ 0.5022,  0.5364,  0.5364,  ..., -1.1932, -1.1418, -1.1589],
            [ 0.7762,  0.7591,  0.7419,  ..., -0.8849, -0.8678, -0.9192],
            [ 0.9474,  0.8789,  0.8618,  ..., -0.0801,  0.0569,  0.0056]],
  
           [[ 0.9930,  0.6604,  0.7654,  ...,  0.8354,  0.4678,  0.9230],
            [ 0.5028,  0.1527,  1.2731,  ...,  0.7304,  0.5203,  0.8880],
            [-0.5651, -0.5301, -0.1625,  ...,  0.8354,  0.5028,  1.0280],
            ...,
            [ 0.5903,  0.6254,  0.6254,  ..., -1.1429, -1.1078, -1.0903],
            [ 0.8704,  0.8529,  0.8354,  ..., -0.7052, -0.7052, -0

In [69]:
model.infer(batch)['cls_feats']

tensor([[ 0.3217, -0.0355, -0.0805,  ..., -0.2298, -0.1458,  0.0090],
        [ 0.3452,  0.0216, -0.0097,  ..., -0.2646, -0.0957, -0.0063]],
       grad_fn=<CatBackward0>)